In [1]:
import gc
from scipy.io import loadmat
from probeinterface import Probe
import spikeinterface as si
import spikeinterface.preprocessing as spre
import spikeinterface.extractors as se
import spikeinterface.sorters as ss
import spikeinterface.curation as scur
from spikeinterface.sortingcomponents.motion import estimate_motion, interpolate_motion

import RCP_analysis as rcp
from RCP_analysis.python.functions.config_loading import *

GEOM_PATH = rcp.resolve_probe_geom_path(PARAMS, REPO_ROOT, session_key=None)
NPRW_CFG     = PARAMS.probes.get("NPRW")
INTAN_STREAM = NPRW_CFG.get("neural_data_stream")
RADII        = (NPRW_CFG.get("local_radius_inner"), NPRW_CFG.get("local_radius_outer"))

global_job_kwargs = dict(n_jobs=PARAMS.parallel_jobs, chunk_duration=PARAMS.chunk)
si.set_global_job_kwargs(**global_job_kwargs)
job_kwargs = dict(n_jobs=-1, progress_bar=True, chunk_duration="1s")

DRIFT_CKPT = OUT_BASE / "checkpoints" / "NPRW_drift"
SORT_CKPT  = OUT_BASE / "checkpoints" / "sorting_results" / "NPRW_concat"
DRIFT_CKPT.mkdir(parents=True, exist_ok=True)
SORT_CKPT.mkdir(parents=True, exist_ok=True)

# 
#     This script preprocesses the Fastigial Intan data.
#     Input:
#         .rhs files from Intan
#     Output:
#         Checkpoint after preprocessing
#         Checkpoint after thresholding and calculating MUA peak locations and firing rate
# 

In [2]:
# Load Geometry
mat_probe = loadmat(Path(GEOM_PATH))
intan_geom = {"x": mat_probe["xcoords"].ravel(), "y": mat_probe["ycoords"].ravel()}
assert intan_geom["x"].size == intan_geom["y"].size
if "chanMap0ind" not in mat_probe:
    raise ValueError("No 0-based chanmap in .mat geometry file.")
intan_probe_mapping = mat_probe["chanMap0ind"].ravel()
if intan_probe_mapping.size != intan_geom["x"].size:
    raise ValueError("device_index_0based length != #contacts")

nprw_probe = Probe(ndim=2)
nprw_probe.set_contacts(
    positions=np.c_[intan_geom["x"], intan_geom["y"]],
    shapes="square", shape_params={"width": 12.0},
)
nprw_probe.set_device_channel_indices(intan_probe_mapping)

In [4]:
BIN_CKPT = OUT_BASE / "checkpoints" / "NPRW_binary"
BIN_CKPT.mkdir(parents=True, exist_ok=True)

fixed_params = dict(
    NSTIM=0, buffer=25, template_leeway=15,
    stim_neural_delay=13, movmean_window=3,
    med_filt_range=25, gauss_filt_range=25, pca_k=3,
)
STIM_STREAM = NPRW_CFG.get("stim_data_stream")

sess = INTAN_ROOT / "BL_closed_loop_STIM_003_250528_140952"
bin_path = BIN_CKPT / f"{sess.name}_neural.bin"

In [5]:
sess = INTAN_ROOT / "BL_closed_loop_STIM_003_250528_145904"
bin_path = BIN_CKPT / f"{sess.name}_neural.bin"

In [ ]:
# load Intan data
rec_raw = se.read_split_intan_files(sess, mode="concatenate", stream_name=INTAN_STREAM, use_names_as_ids=True)
rec_raw = spre.unsigned_to_signed(rec_raw)
rec_raw = rec_raw.set_probe(nprw_probe)
rec_raw = rcp.reorder_recording_to_geometry(rec_raw, intan_probe_mapping)
# Local CMR
rec_raw = spre.bandpass_filter(rec_raw, freq_min=float(PARAMS.highpass_hz), freq_max=(PARAMS.lowpass_hz))
rec_raw = spre.common_reference(rec_raw, reference="local", operator="median", local_radius=(RADII[0], RADII[1]))
print(rec_raw)

ChannelSliceRecording: 128 channels - 30.0kHz - 1 segments - 11,636,480 samples 
                       387.88s (6.46 minutes) - int16 dtype - 2.77 GiB


In [12]:
traces_sample = rec_raw.get_traces(start_frame=0, end_frame=int(30000*60))  # 1 min sample
import numpy as np
stds = traces_sample.std(axis=0)
print(np.where(stds == 0)[0])  # channel indices with zero variance
print(stds.min(), stds.max())

[]
719.0969951533459 7482.724812521952


In [ ]:
import matplotlib.pyplot as plt
plt.ion()
motion, motion_info = spre.compute_motion(
    rec_raw, preset="dredge", folder=DRIFT_CKPT, output_motion_info=True, overwrite=True, **job_kwargs
)

%matplotlib inline
from spikeinterface.widgets import plot_motion, plot_motion_info
fig = plt.figure(figsize=(14, 8))
plot_motion_info(
    motion_info, rec_raw,
    figure=fig,
    depth_lim=(400, 600),
    color_amplitude=True,
    amplitude_cmap="inferno",
    scatter_decimate=10,
)

from spikeinterface.sortingcomponents.motion import interpolate_motion
interpolated_recording = interpolate_motion(
    recording=rec_raw, motion=motion_info['motion'],
    border_mode="force_extrapolate"
)
interpolated_recording

detect and localize (workers: 12 processes fork):   0%|          | 0/388 [00:00<?, ?it/s]

/home/bryan/miniconda3/envs/pipeline/lib/python3.10/site-packages/spikeinterface/postprocessing/localization_tools.py:412: RuntimeWarning: divide by zero encountered in divide
  alpha = (q * wf_data / max_data).sum() / (q * q).sum()
/home/bryan/miniconda3/envs/pipeline/lib/python3.10/site-packages/spikeinterface/postprocessing/localization_tools.py:412: RuntimeWarning: invalid value encountered in divide
  alpha = (q * wf_data / max_data).sum() / (q * q).sum()
/home/bryan/miniconda3/envs/pipeline/lib/python3.10/site-packages/spikeinterface/postprocessing/localization_tools.py:511: RuntimeWarning: divide by zero encountered in scalar divide
  decreasing_data[c] *= decreasing_data[parents_rel].max() / decreasing_data[c]
/home/bryan/miniconda3/envs/pipeline/lib/python3.10/site-packages/spikeinterface/postprocessing/localization_tools.py:414: RuntimeWarning: divide by zero encountered in divide
  np.square(wf_data / max_data - data_at(x, y, z, alpha, local_contact_locations)).mean()
/home/

ValueError: cannot convert float NaN to integer

In [ ]:
# Save the corrected recording to disk so it's serializable for the sorter
saved_rec_path = SORT_CKPT / "rec_corrected_saved"

rec_corrected_saved = rec_corrected.save(
    folder=saved_rec_path,
    overwrite=True,
    **job_kwargs,
)
print(rec_corrected_saved)

/home/bryan/miniconda3/envs/pipeline/lib/python3.10/site-packages/spikeinterface/core/base.py:991: UserWarning: The extractor is not serializable to file. The provenance will not be saved.
  warnings.warn("The extractor is not serializable to file. The provenance will not be saved.")


write_binary_recording 
engine=process - n_jobs=12 - samples_per_chunk=30,000 - chunk_memory=14.65 MiB - total_memory=175.78 MiB - chunk_duration=1.00s


write_binary_recording (workers: 12 processes fork):   0%|          | 0/388 [00:00<?, ?it/s]

BinaryFolderRecording: 128 channels - 30.0kHz - 1 segments - 11,636,480 samples 
                       387.88s (6.46 minutes) - float32 dtype - 5.55 GiB


In [41]:
# Kilosort4
sorting = ss.run_sorter(
    sorter_name="kilosort4",
    recording=rec_corrected_saved,
    folder=SORT_CKPT / "folder_KS4",
    remove_existing_folder=True,
    do_correction=False,
)
sorting = scur.remove_duplicated_spikes(sorting)
sorting = scur.remove_excess_spikes(sorting, rec_corrected_saved)
sorting = scur.remove_redundant_units(sorting, align=False, remove_strategy="max_spikes")
sorting.save(folder=SORT_CKPT / "sorting_curated", overwrite=True)

/home/bryan/miniconda3/envs/pipeline/lib/python3.10/site-packages/torch/cuda/__init__.py:716: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")
100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:10<00:00, 10.31s/it]
/home/bryan/miniconda3/envs/pipeline/lib/python3.10/site-packages/spikeinterface/core/basesorting.py:384: UserWarning: The registered recording will not be persistent on disk, but only available in memory
  warnings.warn("The registered recording will not be persistent on disk, but only available in memory")


NumpyFolder (NumpyFolderSorting): 358 units - 1 segments - 30.0kHz

In [ ]:
# Reload
# sorting = si.load(SORT_CKPT / "sorting_curated")

In [58]:
if 'analyzer_precuration' in dir():
    del analyzer_precuration
gc.collect()
import shutil, pathlib
folder = SORT_CKPT / "analyzer_pre_slay"
if folder.exists():
    shutil.rmtree(folder, ignore_errors=True)
    # force-remove anything left
    import subprocess
    subprocess.run(["rm", "-rf", str(folder)], check=True)
# ── Phy export: post-curation, pre-SLAY ───────────────────────────────────
analyzer_precuration = si.create_sorting_analyzer(
    sorting, rec_corrected_saved,
    format="binary_folder",
    folder=SORT_CKPT / "analyzer_pre_slay",
    overwrite=True,
    **job_kwargs,
)
analyzer_precuration.compute(
    ["random_spikes", "waveforms", "templates", "noise_levels", "spike_amplitudes", "unit_locations"],
    **job_kwargs,
)
analyzer_precuration.compute("isi_histograms")
analyzer_precuration.compute("correlograms", window_ms=100, bin_ms=5.0)
analyzer_precuration.compute(
    "principal_components", n_components=3, mode="by_channel_global", whiten=True, **job_kwargs
)
analyzer_precuration.compute("quality_metrics", metric_names=["snr", "firing_rate"])
analyzer_precuration.compute("template_similarity")


estimate_sparsity (workers: 12 processes fork):   0%|          | 0/388 [00:00<?, ?it/s]

/home/bryan/miniconda3/envs/pipeline/lib/python3.10/site-packages/spikeinterface/core/basesorting.py:384: UserWarning: The registered recording will not be persistent on disk, but only available in memory
  warnings.warn("The registered recording will not be persistent on disk, but only available in memory")


compute_waveforms (workers: 12 processes fork):   0%|          | 0/388 [00:00<?, ?it/s]

Compute : spike_amplitudes (workers: 12 processes fork):   0%|          | 0/388 [00:00<?, ?it/s]

Fitting PCA:   0%|          | 0/358 [00:00<?, ?it/s]

Projecting waveforms:   0%|          | 0/358 [00:00<?, ?it/s]

In [ ]:
import subprocess
result = subprocess.run(["ls", "-la", str(folder)], capture_output=True, text=True)
print(result.stdout)
print(result.stderr)

In [48]:
sorting

UnitsSelectionSorting: 358 units - 1 segments - 30.0kHz

In [51]:
from spikeinterface.exporters import export_to_phy
export_to_phy(
    analyzer_precuration,
    output_folder=SORT_CKPT / "phy_pre_slay",
    remove_if_exists=True,
    **job_kwargs,
)
print(f"Phy export (pre-SLAY) -> {SORT_CKPT / 'phy_pre_slay'}")


write_binary_recording (workers: 12 processes fork):   0%|          | 0/388 [00:00<?, ?it/s]

extract PCs (workers: 12 processes fork):   0%|          | 0/388 [00:00<?, ?it/s]

Run:
phy template-gui  /home/bryan/mnt/cullen_data/Current Project Databases - NHP/2025 Cerebellum prosthesis/Bert/BL_RW_003_Session_1/results/checkpoints/sorting_results/NPRW_concat/phy_pre_slay/params.py
Phy export (pre-SLAY) -> /home/bryan/mnt/cullen_data/Current Project Databases - NHP/2025 Cerebellum prosthesis/Bert/BL_RW_003_Session_1/results/checkpoints/sorting_results/NPRW_concat/phy_pre_slay


In [ ]:

# SLAY on full concatenated sorting
from spikeinterface.curation import compute_merge_unit_groups
analyzer_concat_path = SORT_CKPT / "my_sorting_analyzer_concat"
analyzer_concat = si.create_sorting_analyzer(
    sorting, rec_corrected_saved,
    format="binary_folder", folder=analyzer_concat_path,
    overwrite=True, **job_kwargs,
)